# Лабораторна робота 10.5
## Тема: Мультикласова класифікація з використанням аугментації при завантаженні набору даних з csv файлу (Sign Language MNIST)

**Завдання:**
1. Написати функцію parsing data з CSV.
2. Використати Data Augmentation.
3. Навчити CNN досягати 99% (train) та 95% (val) точності.

In [ ]:
import csv
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from google.colab import files

# Завантажте файли sign_mnist_train.csv та sign_mnist_test.csv з Kaggle або іншого джерела
# https://www.kaggle.com/datamunge/sign-language-mnist
# uploaded = files.upload()

In [ ]:
def get_data(filename):
  # You will need to write code that will read the file passed
  # into this function. The first line contains the header, which you 
  # should ignore. Each successive line contians 785 comma separated values
  # The first value is the label
  # The rest are the pixel values for that picture
  # The function will return 2 np.array types. One with all the labels
  # One with all the images
  #
  # Tips: 
  # If you read a full line (as 'row') then row[0] has the label
  # and row[1:785] has the 784 pixel values
  # Take a look at np.array_split to turn the 784 pixels into 28x28
  # You are reading in strings, but for the output you will want
  # float values. Check out np.array().astype for a conversion
    with open(filename) as training_file:
      ### START CODE HERE
      csv_reader = csv.reader(training_file, delimiter=',')
      first_line = True
      temp_images = []
      temp_labels = []
      for row in csv_reader:
          if first_line:
              first_line = False
          else:
              temp_labels.append(row[0])
              image_data = row[1:785]
              image_data_as_array = np.array_split(image_data, 28)
              temp_images.append(image_data_as_array)
      
      images = np.array(temp_images).astype('float')
      labels = np.array(temp_labels).astype('float')
      ### END CODE HERE
    return images, labels

# Шляхи до файлів (замініть на правильні, якщо потрібно)
training_images, training_labels = get_data('/content/sign_mnist_train.csv')
testing_images, testing_labels = get_data('/content/sign_mnist_test.csv')

print(training_images.shape)
print(training_labels.shape)
print(testing_images.shape)
print(testing_labels.shape)

In [ ]:
# Додаємо розмірність для кольору (хоч це grayscale, Keras потребує 3-го виміру)
training_images = np.expand_dims(training_images, axis=3)
testing_images = np.expand_dims(testing_images, axis=3)

# Create an ImageDataGenerator and do Image Augmentation
train_datagen = ImageDataGenerator(
    ### START CODE HERE
    rescale = 1./255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
    ### END CODE HERE
    )

validation_datagen = ImageDataGenerator(
    ### START CODE HERE
    rescale = 1./255
    ### END CODE HERE
)
    
# Keep These
print(training_images.shape)
print(testing_images.shape)

In [ ]:
# Define the model
# Use no more than 2 Conv2D and 2 MaxPooling2D
model = tf.keras.models.Sequential([
    ### START CODE HERE
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.Dense(26, activation='softmax') # 26 класів (літер)
    ### END CODE HERE
])

# Compile Model. 
model.compile(
    ### START CODE HERE
    optimizer = 'rmsprop',
    loss = 'sparse_categorical_crossentropy', # Sparse бо лейбли - числа, а не one-hot вектори
    metrics = ['accuracy']
    ### END CODE HERE
)

# Train the Model
history = model.fit(
    ### START CODE HERE
    train_datagen.flow(training_images, training_labels, batch_size=32),
    steps_per_epoch=len(training_images) / 32,
    epochs=15,
    validation_data=validation_datagen.flow(testing_images, testing_labels, batch_size=32),
    validation_steps=len(testing_images) / 32
    ### END CODE HERE
)

model.evaluate(testing_images, testing_labels, verbose=0)

In [ ]:
# Plot the chart for accuracy and loss on both training and validation
import matplotlib.pyplot as plt
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs = range(len(acc))

plt.plot(epochs, acc, 'r', label='Training accuracy')
plt.plot(epochs, val_acc, 'b', label='Validation accuracy')
plt.title('Training and validation accuracy')
plt.legend()
plt.figure()

plt.plot(epochs, loss, 'r', label='Training Loss')
plt.plot(epochs, val_loss, 'b', label='Validation Loss')
plt.title('Training and validation loss')
plt.legend()

plt.show()